# Multi-judge analysis — Miami Life metadata tiers (v3 / v4)

Reads score artifacts under `experiments/data/scores/` and asks what the **cross-family judge panel** actually supports.

| Instrument | What it measures |
|---|---|
| Completeness / bibliographic | Structural / easy fields (not LLM-judged) |
| Claim precision (FActScore-style) | Descriptive faithfulness vs OCR |
| Pairwise win rates | Holistic cataloger preference |
| Judge agreement (κ) | Do self / Opus / GPT-5.5 agree? |

**Judges:** `self` = gpt-5.2 (generator), `opus` = Claude Opus 4.8 (cross-family), `gpt55` = GPT-5.5 (same-vendor check).

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCORES = Path("../data/scores").resolve()
assert SCORES.is_dir(), SCORES

TIERS = ["C-simple", "C-standard", "C-precedent"]
JUDGES = ["self", "opus", "gpt55"]
JUDGE_LABEL = {
    "self": "gpt-5.2 (self)",
    "opus": "Opus 4.8",
    "gpt55": "GPT-5.5",
}
TIER_LABEL = {
    "C-simple": "baseline\n(C-simple)",
    "C-standard": "+authority\n(C-standard)",
    "C-precedent": "+peer exp.\n(C-precedent)",
}
COLORS = {
    "self": "#4C566A",
    "opus": "#BF616A",
    "gpt55": "#5E81AC",
    "C-simple": "#88C0D0",
    "C-standard": "#81A1C1",
    "C-precedent": "#5E81AC",
    "tie": "#D8DEE9",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

def load(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

print("SCORES =", SCORES)

SCORES = <path>


## 1. Load claim precision (per card → tier means)

Summary JSONs for `self` omit `C-simple`; we recompute from per-card claim files so all three tiers are comparable.

In [2]:
def claim_dirs(version: str) -> dict[str, Path]:
    if version == "v3":
        return {
            "self": SCORES.parent / "_run3_full_2026-07-04" / "scores" / "claims",
            "opus": SCORES / "claims_opus_v3",
            "gpt55": SCORES / "claims_gpt55_v3",
        }
    return {
        "self": SCORES / "claims",
        "opus": SCORES / "claims_opus",
        "gpt55": SCORES / "claims_gpt55",
    }

def load_claim_cards(version: str) -> pd.DataFram<path>    rows = []
    for judge, d in claim_dirs(version).items():
        if not d.is_dir():
            continue
        for p in d.glob("*__*.json"):
            c = load(p)
            if c.get("claim_precision") is Non<path>                continue
            rows.append({
                "version": version,
                "judge": judge,
                "tier": c.get("tier") or p.name.split("__", 1)[0],
                "doc_id": c["doc_id"],
                "claim_precision": c["claim_precision"],
                "n_claims": c["n_claims"],
                "n_supported": c["n_supported"],
                "n_contradicted": c.get("n_contradicted", 0),
            })
    return pd.DataFrame(rows)

claims = pd.concat([load_claim_cards("v4"), load_claim_cards("v3")], ignore_index=True)
claim_summary = (
    claims.groupby(["version", "judge", "tier"], as_index=False)
    .agg(
        n=("doc_id", "count"),
        claim_precision=("claim_precision", "mean"),
        claims_per_card=("n_claims", "mean"),
        supported_per_card=("n_supported", "mean"),
        contradicted_total=("n_contradicted", "sum"),
    )
)
claim_summary["claim_precision"] = claim_summary["claim_precision"].round(4)
display(claim_summary.pivot_table(
    index=["version", "tier"], columns="judge", values="claim_precision"
).reindex(columns=JUDGES))

judge                  self    opus   gpt55
version tier                               
v3      C-precedent  0.9773  0.9930  0.9559
        C-simple     0.9761     NaN     NaN
        C-standard   0.9789  0.9921  0.9587
v4      C-precedent  0.9680  0.9890  0.9484
        C-simple     0.9761  0.9908  0.9589
        C-standard   0.9724  0.9868  0.9486

In [3]:
def plot_claim_precision(version: str, ax=None):
    sub = claim_summary[claim_summary["version"] == version]
    tiers = [t for t in TIERS if t in set(sub["tier"])]
    x = np.arange(len(tiers))
    width = 0.25
    ax = ax or plt.gca()
    for i, j in enumerate(JUDGES):
        vals = [
            float(sub[(sub["tier"] == t) & (sub["judge"] == j)]["claim_precision"].iloc[0])
            if not sub[(sub["tier"] == t) & (sub["judge"] == j)].empty else np.nan
            for t in tiers
        ]
        ax.bar(x + (i - 1) * width, vals, width, label=JUDGE_LABEL[j], color=COLORS[j])
    ax.set_xticks(x)
    ax.set_xticklabels([TIER_LABEL[t] for t in tiers])
    ax.set_ylim(0.90, 1.005)
    ax.set_ylabel("Claim precision (supported / total)")
    ax.set_title(f"{version}: descriptive claim precision by tier × judge")
    ax.legend(frameon=False, loc="lower right")
    return ax

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
plot_claim_precision("v4", axes[0])
plot_claim_precision("v3", axes[1])
axes[1].set_ylabel("")
fig.tight_layout()
plt.show()

print("Reading note: y-axis starts at 0.90 — all tiers are already high; look at small gaps, not absolute height.")

Reading note: y-axis starts at 0.90 — all tiers are already high; look at small gaps, not absolute height.


## 2. Pairwise win rates (holistic preference)

Each cell is one (doc, tier-pair). Winner after A/B order-swap; disagreement → tie.

In [4]:
def pairwise_path(version: str, judge: str) -> Pat<path>    if version == "v3":
        if judge == "self":
            return SCORES.parent / "_run3_full_2026-07-04" / "scores" / "pilot_pairwise_descriptive.json"
        return SCORES / f"pilot_pairwise_descriptive_{judge}_v3.json"
    if judge == "self":
        return SCORES / "pilot_pairwise_descriptive.json"
    return SCORES / f"pilot_pairwise_descriptive_{judge}.json"

def load_pairwise_summary(version: str) -> pd.DataFram<path>    rows = []
    for judge in JUDGE<path>        p = pairwise_path(version, judge)
        if not p.exists():
            continue
        summary = load(p).get("summary") or {}
        for key, s in summary.items():
            # key like C-simple_vs_C-standard
            a, b = key.split("_vs_")
            n = s.get("n", 0) or 0
            rows.append({
                "version": version,
                "judge": judge,
                "pair": key,
                "tier_a": a,
                "tier_b": b,
                "wins_a": s.get(a, 0),
                "wins_b": s.get(b, 0),
                "tie": s.get("tie", 0),
                "n": n,
                "pct_a": 100 * s.get(a, 0) / n if n else np.nan,
                "pct_b": 100 * s.get(b, 0) / n if n else np.nan,
                "pct_tie": 100 * s.get("tie", 0) / n if n else np.nan,
            })
    return pd.DataFrame(rows)

pair = pd.concat([load_pairwise_summary("v4"), load_pairwise_summary("v3")], ignore_index=True)
display(pair[pair["version"] == "v4"][["judge", "pair", "wins_a", "wins_b", "tie", "n", "pct_a", "pct_b", "pct_tie"]]
        .sort_values(["pair", "judge"]))

,judge,pair,wins_a,wins_b,tie,n,pct_a,pct_b,pct_tie
7,gpt55,C-simple_vs_C-precedent,141,34,28,203,69.458128,16.748768,13.793103
4,opus,C-simple_vs_C-precedent,136,10,57,203,66.995074,4.926108,28.078818
1,self,C-simple_vs_C-precedent,29,16,158,203,14.285714,7.881773,77.832512
6,gpt55,C-simple_vs_C-standard,136,38,29,203,66.995074,18.719212,14.285714
3,opus,C-simple_vs_C-standard,119,15,69,203,58.620690,7.389163,33.990148
0,self,C-simple_vs_C-standard,25,18,160,203,12.315271,8.866995,78.817734
8,gpt55,C-standard_vs_C-precedent,65,65,73,203,32.019704,32.019704,35.960591
5,opus,C-standard_vs_C-precedent,56,53,94,203,27.586207,26.108374,46.305419
2,self,C-standard_vs_C-precedent,12,10,181,203,5.911330,4.926108,89.162562


In [ ]:
def plot_pairwise_stacked(version: str):
    sub = pair[pair["version"] == version].copy()
    # only pairs scored by ≥2 judges (v3 self has extra C-simple pairs Opus/GPT-5.5 lack)
    counts = sub.groupby("pair")["judge"].nunique()
    pairs = [p for p in dict.fromkeys(sub["pair"]) if counts.get(p, 0) >= 2]
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.2 * len(pairs), 4.5), sharey=True)
    if len(pairs) == 1:
        axes = [axes]
    for ax, pname in zip(axes, pairs):
        block = sub[sub["pair"] == pname]
        judges = [j for j in JUDGES if j in set(block["judge"])]
        a = block["tier_a"].iloc[0]
        b = block["tier_b"].iloc[0]
        x = np.arange(len(judges))
        pct_a = [float(block[block["judge"] == j]["pct_a"].iloc[0]) for j in judges]
        pct_b = [float(block[block["judge"] == j]["pct_b"].iloc[0]) for j in judges]
        pct_t = [float(block[block["judge"] == j]["pct_tie"].iloc[0]) for j in judges]
        ax.bar(x, pct_a, color=COLORS[a], label=a)
        ax.bar(x, pct_b, bottom=pct_a, color=COLORS[b], label=b)
        ax.bar(x, pct_t, bottom=np.array(pct_a) + np.array(pct_b), color=COLORS["tie"], label="tie")
        ax.set_xticks(x)
        ax.set_xticklabels([JUDGE_LABEL[j].replace(" ", "\n") for j in judges], fontsize=9)
        ax.set_ylim(0, 100)
        ax.set_title(f"{a}  vs  {b}")
        if ax is axes[0]:
            ax.set_ylabel("% of issues")
        ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.suptitle(f"{version}: pairwise descriptive preference (order-swapped)", y=1.02)
    fig.tight_layout()
    plt.show()

plot_pairwise_stacked("v4")
plot_pairwise_stacked("v3")

## 3. Cross-judge agreement

From `judge_agreement*.json`: raw agreement % and Cohen's κ. Claim κ is depressed by class imbalance (most claims = supported).

In [6]:
def load_agreement(version: str) -> dic<path>    p = SCORES / ("judge_agreement_v3.json" if version == "v3" else "judge_agreement.json")
    return load(p)

def agreement_tables(version: str):
    ag = load_agreement(version)
    # pairwise κ
    prow = []
    for pname, block in (ag["pairwise"].get("by_pair") or {}).items():
        for key, kappa in (block.get("cohen_kappa") or {}).items():
            prow.append({
                "pair": pname,
                "judges": key,
                "agreement_pct": block["agreement_pct"][key],
                "cohen_kappa": kappa,
                "fleiss": block.get("fleiss_kappa"),
            })
    pair_df = pd.DataFrame(prow)

    crow = []
    cl = ag["claims"]
    for key in cl.get("cohen_kappa_3way") or {}:
        crow.append({
            "judges": key,
            "agree_3way": cl["agreement_pct"][key],
            "kappa_3way": cl["cohen_kappa_3way"][key],
            "agree_binary": cl["agreement_pct_binary"][key],
            "kappa_binary": cl["cohen_kappa_binary"][key],
        })
    claim_df = pd.DataFrame(crow)
    return pair_df, claim_df, cl

for ver in ("v4", "v3"):
    print(f"\n=== {ver} pairwise agreement ===")
    pdf, cdf, cl = agreement_tables(ver)
    display(pdf)
    print(f"{ver} claim agreement (n_claims={cl.get('n_shared_claims')})")
    display(cdf)
    print("Fleiss κ claims 3-way / binary:",
          cl.get("fleiss_kappa_3way"), cl.get("fleiss_kappa_binary"))


=== v4 pairwise agreement ===


,pair,judges,agreement_pct,cohen_kappa,fleiss
0,C-simple_vs_C-precedent,self_vs_opus,36.45,0.0680,0.0011
1,C-simple_vs_C-precedent,self_vs_gpt55,28.08,0.0782,0.0011
2,C-simple_vs_C-precedent,opus_vs_gpt55,62.07,0.2222,0.0011
3,C-simple_vs_C-standard,self_vs_opus,42.36,0.1179,0.0295
4,C-simple_vs_C-standard,self_vs_gpt55,28.57,0.0939,0.0295
5,C-simple_vs_C-standard,opus_vs_gpt55,55.67,0.1863,0.0295
6,C-standard_vs_C-precedent,self_vs_opus,52.71,0.1524,0.1022
7,C-standard_vs_C-precedent,self_vs_gpt55,43.35,0.1212,0.1022
8,C-standard_vs_C-precedent,opus_vs_gpt55,47.29,0.2032,0.1022


v4 claim agreement (n_claims=33450)


,judges,agree_3way,kappa_3way,agree_binary,kappa_binary
0,self_vs_opus,96.84,0.1602,96.86,0.1657
1,self_vs_gpt55,93.64,0.1436,93.67,0.1459
2,opus_vs_gpt55,94.68,0.0992,94.69,0.0998


Fleiss κ claims 3-way / binary: 0.128 0.1305

=== v3 pairwise agreement ===


,pair,judges,agreement_pct,cohen_kappa,fleiss
0,C-standard_vs_C-precedent,self_vs_opus,43.35,-0.0029,0.0802
1,C-standard_vs_C-precedent,self_vs_gpt55,42.86,0.1102,0.0802
2,C-standard_vs_C-precedent,opus_vs_gpt55,52.22,0.2744,0.0802


v3 claim agreement (n_claims=26328)


,judges,agree_3way,kappa_3way,agree_binary,kappa_binary
0,self_vs_opus,97.56,0.1254,97.58,0.1336
1,self_vs_gpt55,94.63,0.1285,94.67,0.1320
2,opus_vs_gpt55,95.41,0.0606,95.44,0.0653


Fleiss κ claims 3-way / binary: 0.0997 0.1047


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: pairwise κ self vs opus / opus vs gpt55 for each pair (v4)
pdf, _, _ = agreement_tables("v4")
focus = pdf[pdf["judges"].isin(["self_vs_opus", "opus_vs_gpt55", "self_vs_gpt55"])]
pairs = list(dict.fromkeys(focus["pair"]))
x = np.arange(len(pairs))
width = 0.25
for i, key in enumerate(["self_vs_opus", "self_vs_gpt55", "opus_vs_gpt55"]):
    vals = [float(focus[(focus["pair"] == p) & (focus["judges"] == key)]["cohen_kappa"].iloc[0])
            for p in pairs]
    axes[0].bar(x + (i - 1) * width, vals, width, label=key.replace("_vs_", " ↔ "))
axes[0].axhline(0, color="#333", lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([p.replace("_vs_", "\nvs\n") for p in pairs], fontsize=8)
axes[0].set_ylabel("Cohen's κ")
axes[0].set_title("v4 pairwise: judge agreement (κ)")
axes[0].legend(frameon=False, fontsize=8)

# right: claim agreement %
_, cdf, _ = agreement_tables("v4")
x2 = np.arange(len(cdf))
axes[1].bar(x2 - 0.15, cdf["agree_3way"], 0.3, label="3-way labels", color="#5E81AC")
axes[1].bar(x2 + 0.15, cdf["agree_binary"], 0.3, label="binary supported?", color="#88C0D0")
axes[1].set_xticks(x2)
axes[1].set_xticklabels([j.replace("_vs_", "\n↔\n") for j in cdf["judges"]], fontsize=8)
axes[1].set_ylim(90, 100)
axes[1].set_ylabel("Agreement %")
axes[1].set_title("v4 claim-verify: raw agreement (high; κ still low)")
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()
plt.show()

## 4. Structural baselines (not LLM-judged)

Completeness and bibliographic match vs librarian MODS — these do **not** depend on the judge panel.

In [8]:
comp = load(SCORES / "pilot_completeness.json")
biblio = load(SCORES / "bibliographic_vs_librarian.json")["per_tier"]

struct = pd.DataFrame([
    {
        "tier": t,
        "completeness": comp[t]["overall"],
        "required_completeness": comp[t]["required"],
        "date_exact_rate": biblio[t]["date_exact"] / biblio[t]["n"],
        "creator_surname_rate": biblio[t]["creator_surname"] / biblio[t]["n"],
        "title_overlap_rate": biblio[t]["title_overlap"] / biblio[t]["n"],
    }
    for t in TIERS
])
display(struct.round(4))

fig, ax = plt.subplots(figsize=(7.5, 4))
metrics = ["completeness", "date_exact_rate", "creator_surname_rate"]
x = np.arange(len(TIERS))
width = 0.25
for i, m in enumerate(metrics):
    ax.bar(x + (i - 1) * width, struct[m], width, label=m)
ax.set_xticks(x)
ax.set_xticklabels([TIER_LABEL[t] for t in TIERS])
ax.set_ylim(0.5, 1.05)
ax.set_ylabel("Rate")
ax.set_title("v4 structural metrics (deterministic / librarian compare)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

,tier,completeness,required_completeness,date_exact_rate,creator_surname_rate,title_overlap_rate
0,C-simple,0.9212,0.9862,0.9852,0.7291,1.0
1,C-standard,0.9379,0.9857,0.9901,0.7094,1.0
2,C-precedent,0.9356,0.9847,0.9852,0.7192,1.0


## 4b. Structure metrics — the headline deterministic table (see doc `09`)

Both LLM instruments strip authority URIs before judging (`judge.py::_strip_authority_uris`), so
the knowledge tiers' main deliverable is **invisible** to §1–§2. This section measures it
directly from the v4 cards and the rejection ledger — zero API calls, zero judge dependence.

- **Authority links**: URIs (id.loc.gov / GeoNames) embedded in card fields.
- **Rejection ledger**: candidate links refused by the identity gate (match-score thresholds +
  lifespan sanity check), deduped by (surface, URI) since the ledger accumulates across runs.
- Links originate from live id.loc.gov / GeoNames API responses (see `authority_cache.json`),
  so they are resolvable by construction.

In [9]:
import re
from collections import Counter

CARDS = SCORES.parent / "cards"
URI_RE = re.compile(r"https?://(?:id\.loc\.gov|www\.geonames\.org)[^\s()\"]+")

def _walk_uris(x):
    if isinstance(x, str):
        yield from URI_RE.findall(x)
    elif isinstance(x, list):
        for v in <path>            yield from _walk_uris(v)
    elif isinstance(x, dict):
        for v in x.values():
            yield from _walk_uris(v)

rows = []
for tier in TIER<path>    total, per_card, uniq, domains = 0, [], set(), Counter()
    for p in sorted((CARDS / tier).glob("*.json")):
        uris = list(_walk_uris(load(p).get("fields", {})))
        total += len(uris)
        per_card.append(len(uris))
        uniq.update(uris)
        for u in uri<path>            domains["LoC" if "id.loc.gov" in u else "GeoNames"] += 1
    n = len(per_card)
    rows.append({
        "tier": tier,
        "cards": n,
        "authority_links_total": total,
        "links_per_card": round(total / n, 1) if n else 0,
        "cards_with_links": sum(1 for c in per_card if c),
        "unique_URIs": len(uniq),
        "LoC": domains["LoC"],
        "GeoNames": domains["GeoNames"],
    })
structure_links = pd.DataFrame(rows)
display(structure_links)

# Rejection ledger (identity gate at work)
ledger = [json.loads(l) for l in
          (SCORES.parent / "authority_rejections.jsonl").read_text(encoding="utf-8").strip().splitlines()]
uniq_rej = {(l["surface"], l["uri"]): l for l in ledger}
reason_kind = Counter(
    "lifespan conflict" if l["reason"].startswith("lifespan") else "match score below threshold"
    for l in uniq_rej.values()
)
print(f"Rejection ledger: {len(ledger)} events, {len(uniq_rej)} unique (surface, URI) rejections")
for k, v in reason_kind.most_common():
    print(f"  {k}: {v}")

accepted = int(structure_links.loc[structure_links["tier"] == "C-standard", "unique_URIs"].iloc[0])
print(f"\nIdentity gate: {accepted} unique URIs accepted vs {len(uniq_rej)} unique candidates rejected "
      f"({len(uniq_rej) / (accepted + len(uniq_rej)):.0%} of unique candidates refused)")

,tier,cards,authority_links_total,links_per_card,cards_with_links,unique_URIs,LoC,GeoNames
0,C-simple,203,0,0.0,0,0,0,0
1,C-standard,203,2837,14.0,202,521,1923,914
2,C-precedent,203,2810,13.8,202,521,1882,928


Rejection ledger: 844 events, 361 unique (surface, URI) rejections
  match score below threshold: 324
  lifespan conflict: 37

Identity gate: 521 unique URIs accepted vs 361 unique candidates rejected (41% of unique candidates refused)


In [10]:
# The R1 headline figure: structure added (deterministic) vs faithfulness cost (3 judges)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

x = np.arange(len(TIERS))
axes[0].bar(x, structure_links["links_per_card"], 0.55,
            color=[COLORS[t] for t in TIERS])
for xi, v in zip(x, structure_links["links_per_card"]):
    axes[0].text(xi, v + 0.25, f"{v:.1f}", ha="center", fontsize=10)
axes[0].set_xticks(x)
axes[0].set_xticklabels([TIER_LABEL[t] for t in TIERS])
axes[0].set_ylabel("Verified authority links per card")
axes[0].set_title("Structure added (deterministic count)")

sub = claim_summary[claim_summary["version"] == "v4"]
width = 0.25
for i, j in enumerate(JUDGES):
    vals = [float(sub[(sub["tier"] == t) & (sub["judge"] == j)]["claim_precision"].iloc[0])
            for t in TIERS]
    axes[1].bar(x + (i - 1) * width, vals, width, label=JUDGE_LABEL[j], color=COLORS[j])
axes[1].set_xticks(x)
axes[1].set_xticklabels([TIER_LABEL[t] for t in TIERS])
axes[1].set_ylim(0.90, 1.005)
axes[1].set_ylabel("Claim precision")
axes[1].set_title("Faithfulness cost of that structure: none")
axes[1].legend(frameon=False, fontsize=8, loc="lower right")

fig.suptitle("R1 — structure for free: +14 links/card at flat descriptive faithfulness", y=1.04)
fig.tight_layout()
plt.show()

## 5. Does G0→G1 / G1→G2 survive an independent judge?

Define simple deltas on claim precision (same cards, different knowledge):
- **G0→G1 effect:** `C-standard − C-simple`
- **G1→G2 effect:** `C-precedent − C-standard`

In [11]:
def tier_delta(version: str) -> pd.DataFram<path>    sub = claim_summary[claim_summary["version"] == version]
    rows = []
    for j in JUDGE<path>        s = sub[sub["judge"] == j].set_index("tier")["claim_precision"]
        if not {"C-simple", "C-standard"}.issubset(s.index):
            g01 = np.nan
        els<path>            g01 = s["C-standard"] - s["C-simple"]
        if not {"C-standard", "C-precedent"}.issubset(s.index):
            g12 = np.nan
        els<path>            g12 = s["C-precedent"] - s["C-standard"]
        rows.append({"judge": j, "G0→G1 (std−simple)": g01, "G1→G2 (prec−std)": g12})
    return pd.DataFrame(rows)

print("Claim-precision deltas (percentage points)")
for ver in ("v4", "v3"):
    print(f"\n{ver}")
    d = tier_delta(ver)
    d_disp = d.copy()
    d_disp["judge"] = d_disp["judge"].map(JUDGE_LABEL)
    display(d_disp.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, ver in zip(axes, ("v4", "v3")):
    d = tier_delta(ver)
    x = np.arange(len(JUDGES))
    ax.bar(x - 0.15, d["G0→G1 (std−simple)"], 0.3, label="G0→G1", color="#81A1C1")
    ax.bar(x + 0.15, d["G1→G2 (prec−std)"], 0.3, label="G1→G2", color="#BF616A")
    ax.axhline(0, color="#333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([JUDGE_LABEL[j].replace(" ", "\n") for j in JUDGES], fontsize=8)
    ax.set_title(f"{ver}: claim-precision deltas")
    ax.set_ylabel("Δ precision" if ax is axes[0] else "")
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

Claim-precision deltas (percentage points)

v4


,judge,G0→G1 (std−simple),G1→G2 (prec−std)
0,gpt-5.2 (self),-0.0037,-0.0044
1,Opus 4.8,-0.0040,0.0022
2,GPT-5.5,-0.0103,-0.0002



v3


,judge,G0→G1 (std−simple),G1→G2 (prec−std)
0,gpt-5.2 (self),0.0028,-0.0016
1,Opus 4.8,NaN,0.0009
2,GPT-5.5,NaN,-0.0028


In [12]:
# Pairwise: net preference for knowledge tier over baseline (exclude ties)
def net_pref(version: str, pair_name: str, fav: str, other: str) -> pd.DataFram<path>    sub = pair[(pair["version"] == version) & (pair["pair"] == pair_name)]
    rows = []
    for _, r in sub.iterrows():
        wa = r["wins_a"] if r["tier_a"] == fav else r["wins_b"]
        wb = r["wins_b"] if r["tier_a"] == fav else r["wins_a"]
        # if fav is tier_b, swap
        if r["tier_a"] == other and r["tier_b"] == fa<path>            wa, wb = r["wins_b"], r["wins_a"]
        elif r["tier_a"] == fav and r["tier_b"] == othe<path>            wa, wb = r["wins_a"], r["wins_b"]
        decided = wa + wb
        rows.append({
            "judge": r["judge"],
            "fav_wins": wa,
            "other_wins": wb,
            "tie": r["tie"],
            "fav_share_of_decided": wa / decided if decided else np.nan,
            "tie_rate": r["pct_tie"] / 100,
        })
    return pd.DataFrame(rows)

print("Among non-tie decisions, share won by the FIRST named tier")
for pair_name, fav, other in [
    ("C-simple_vs_C-standard", "C-simple", "C-standard"),
    ("C-simple_vs_C-precedent", "C-simple", "C-precedent"),
    ("C-standard_vs_C-precedent", "C-standard", "C-precedent"),
]:
    print(f"\nv4 {pair_name} — fav={fav}")
    d = net_pref("v4", pair_name, fav, other)
    d["judge"] = d["judge"].map(JUDGE_LABEL)
    display(d.round(3))

Among non-tie decisions, share won by the FIRST named tier

v4 C-simple_vs_C-standard — fav=C-simple


,judge,fav_wins,other_wins,tie,fav_share_of_decided,tie_rate
0,gpt-5.2 (self),25,18,160,0.581,0.788
1,Opus 4.8,119,15,69,0.888,0.340
2,GPT-5.5,136,38,29,0.782,0.143



v4 C-simple_vs_C-precedent — fav=C-simple


,judge,fav_wins,other_wins,tie,fav_share_of_decided,tie_rate
0,gpt-5.2 (self),29,16,158,0.644,0.778
1,Opus 4.8,136,10,57,0.932,0.281
2,GPT-5.5,141,34,28,0.806,0.138



v4 C-standard_vs_C-precedent — fav=C-standard


,judge,fav_wins,other_wins,tie,fav_share_of_decided,tie_rate
0,gpt-5.2 (self),12,10,181,0.545,0.892
1,Opus 4.8,56,53,94,0.514,0.463
2,GPT-5.5,65,65,73,0.500,0.360


## 5b. Statistical tests (bootstrap CIs + significance)

Means alone are not enough for the paper. This section add<path>
1. **Paired bootstrap 95% CI** on claim-precision deltas (same `doc_id` across tiers) — tests R1's "flat faithfulness" and the dropped G0→G1 claim.
2. **Two-sided binomial test** on pairwise decided votes (H₀: P(fav)=0.5) + Wilson 95% CI — tests R2/R4 preference and R3's G1↔G2 coin-flip.
3. **Bootstrap 95% CI on Cohen's κ** for pairwise judge pairs — tests R2's "self ≠ independent" gap.

All local; no API calls. Seed fixed for reproducibility.

In [ ]:
from scipy import stats

RNG = np.random.default_rng(20260710)
N_BOOT = 5000

def bootstrap_ci(values: np.ndarray, n_boot: int = N_BOOT, alpha: float = 0.05):
    """Percentile bootstrap CI for the mean of an array."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, (np.nan, np.nan)
    means = []
    n = len(values)
    for _ in range(n_boot):
        means.append(values[RNG.integers(0, n, n)].mean())
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(values.mean()), (float(lo), float(hi))

def paired_delta_ci(a: pd.Series, b: pd.Series, n_boot: int = N_BOOT, alpha: float = 0.05):
    """Paired bootstrap CI for mean(a − b), aligned on index (doc_id)."""
    joined = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    diffs = (joined["a"] - joined["b"]).to_numpy(dtype=float)
    mean, (lo, hi) = bootstrap_ci(diffs, n_boot=n_boot, alpha=alpha)
    # also a paired Wilcoxon signed-rank (two-sided) when n≥10 and not all-zero
    if len(diffs) >= 10 and np.any(diffs != 0):
        tr<path>            w = stats.wilcoxon(diffs, zero_method="wilcox", alternative="two-sided")
            p = float(w.pvalue)
        except ValueErro<path>            p = np.nan
    els<path>        p = np.nan
    return {
        "n_paired": len(diffs),
        "mean_delta": mean,
        "ci95_lo": lo,
        "ci95_hi": hi,
        "ci_excludes_0": bool(lo > 0 or hi < 0) if np.isfinite(lo) else False,
        "wilcoxon_p": p,
    }

def claim_delta_tests(version: str) -> pd.DataFram<path>    sub = claims[claims["version"] == version]
    rows = []
    for j in JUDGE<path>        sj = sub[sub["judge"] == j]
        piv = sj.pivot_table(index="doc_id", columns="tier", values="claim_precision")
        for label, hi, lo in [
            ("G0→G1 (std−simple)", "C-standard", "C-simple"),
            ("G1→G2 (prec−std)", "C-precedent", "C-standard"),
        ]:
            if hi not in piv.columns or lo not in piv.column<path>                continue
            r = paired_delta_ci(piv[hi], piv[lo])
            r.update({"version": version, "judge": j, "contrast": label})
            rows.append(r)
    return pd.DataFrame(rows)

claim_tests = pd.concat([claim_delta_tests("v4"), claim_delta_tests("v3")], ignore_index=True)
print("Claim-precision paired deltas — bootstrap 95% CI + Wilcoxon p")
disp = claim_tests.copy()
disp["judge"] = disp["judge"].map(JUDGE_LABEL)
disp["mean_delta"] = disp["mean_delta"].map(lambda x: f"{x:+.4f}")
disp["ci95"] = disp.apply(lambda r: f"[{r['ci95_lo']:+.4f}, {r['ci95_hi']:+.4f}]", axis=1)
disp["wilcoxon_p"] = disp["wilcoxon_p"].map(lambda x: f"{x:.3g}" if pd.notna(x) else "—")
display(disp[["version", "judge", "contrast", "n_paired", "mean_delta", "ci95",
              "ci_excludes_0", "wilcoxon_p"]])

print("\nReading note: if CI includes 0 → cannot reject 'no precision change' (supports R1 flatness).")
print("ci_excludes_0=True would be needed to revive the old G0→G1 quality-lift claim.")

Claim-precision paired deltas — bootstrap 95% CI + Wilcoxon p


,version,judge,contrast,n_paired,mean_delta,ci95,ci_excludes_0,wilcoxon_p
0,v4,gpt-5.2 (self),G0→G1 (std−simple),203,-0.0038,"[-0.0094, +0.0017]",False,0.142
1,v4,gpt-5.2 (self),G1→G2 (prec−std),203,-0.0043,"[-0.0101, +0.0014]",False,0.2
2,v4,Opus 4.8,G0→G1 (std−simple),203,-0.0041,"[-0.0073, -0.0009]",True,0.00759
3,v4,Opus 4.8,G1→G2 (prec−std),203,+0.0022,"[-0.0010, +0.0055]",False,0.292
4,v4,GPT-5.5,G0→G1 (std−simple),203,-0.0104,"[-0.0254, +0.0050]",False,0.00986
5,v4,GPT-5.5,G1→G2 (prec−std),203,-0.0002,"[-0.0126, +0.0114]",False,0.57
6,v3,gpt-5.2 (self),G0→G1 (std−simple),203,+0.0027,"[-0.0024, +0.0078]",False,0.421
7,v3,gpt-5.2 (self),G1→G2 (prec−std),203,-0.0015,"[-0.0062, +0.0030]",False,0.576
8,v3,Opus 4.8,G1→G2 (prec−std),203,+0.0009,"[-0.0013, +0.0031]",False,0.597
9,v3,GPT-5.5,G1→G2 (prec−std),203,-0.0028,"[-0.0158, +0.0106]",False,0.535



Reading note: if CI includes 0 → cannot reject 'no precision change' (supports R1 flatness).
ci_excludes_0=True would be needed to revive the old G0→G1 quality-lift claim.


In [ ]:
def load_pairwise_results(version: str) -> pd.DataFram<path>    """Per-issue pairwise winners (not just summary counts)."""
    rows = []
    for judge in JUDGE<path>        p = pairwise_path(version, judge)
        if not p.exists():
            continue
        payload = load(p)
        for r in payload.get("results") or []:
            a, b = r["pair"]
            rows.append({
                "version": version,
                "judge": judge,
                "doc_id": r["doc_id"],
                "pair": f"{a}_vs_{b}",
                "tier_a": a,
                "tier_b": b,
                "winner": r.get("winner"),  # tier name or "tie"
            })
    return pd.DataFrame(rows)

def wilson_ci(k: int, n: int, alpha: float = 0.05):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2)
    phat = k / n
    denom = 1 + z**2 / n
    centre = (phat + z**2 / (2 * n)) / denom
    half = (z / denom) * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))
    return (float(max(0, centre - half)), float(min(1, centre + half)))

def pairwise_binomial(version: str) -> pd.DataFram<path>    raw = load_pairwise_results(version)
    rows = []
    for (judge, pname), g in raw.groupby(["judge", "pair"]):
        a = g["tier_a"].iloc[0]
        b = g["tier_b"].iloc[0]
        # for each named fav, test whether fav beats other among decided
        for fav, other in [(a, b), (b, a)]:
            wins_fav = int((g["winner"] == fav).sum())
            wins_other = int((g["winner"] == other).sum())
            ties = int((g["winner"] == "tie").sum())
            decided = wins_fav + wins_other
            if decided == 0:
                continue
            # two-sided exact binomial: H0 P(fav)=0.5 among decided
            p = float(stats.binomtest(wins_fav, decided, 0.5, alternative="two-sided").pvalue)
            lo, hi = wilson_ci(wins_fav, decided)
            rows.append({
                "version": version,
                "judge": judge,
                "pair": pname,
                "fav": fav,
                "other": other,
                "wins_fav": wins_fav,
                "wins_other": wins_other,
                "tie": ties,
                "n_decided": decided,
                "fav_share": wins_fav / decided,
                "wilson_lo": lo,
                "wilson_hi": hi,
                "binom_p": p,
                "sig_vs_coin": p < 0.05,
            })
    return pd.DataFrame(rows)

pw_tests = pd.concat([pairwise_binomial("v4"), pairwise_binomial("v3")], ignore_index=True)

# Deduplicate: keep one row per (version, judge, pair) with fav = tier_a (canonical order)
# Actually we want both directions is redundant — keep fav=first named in pair for display,
# plus a focused view for the paper contrasts.
focus_contrasts = [
    ("C-simple_vs_C-standard", "C-simple"),   # R2/R4: does baseline win?
    ("C-simple_vs_C-precedent", "C-simple"),
    ("C-standard_vs_C-precedent", "C-standard"),  # R3: is G1↔G2 a coin flip?
]

focus = []
for ver in ("v4", "v3"):
    for pname, fav in focus_contrast<path>        block = pw_tests[(pw_tests["version"] == ver) & (pw_tests["pair"] == pname) & (pw_tests["fav"] == fav)]
        focus.append(block)
focus_df = pd.concat(focus, ignore_index=True) if focus else pd.DataFrame()

print("Pairwise decided-vote tests — H₀: P(fav)=½  |  Wilson 95% CI on fav share")
disp = focus_df.copy()
disp["judge"] = disp["judge"].map(JUDGE_LABEL)
disp["fav_share"] = disp["fav_share"].map(lambda x: f"{x:.1%}")
disp["wilson"] = disp.apply(lambda r: f"[{r['wilson_lo']:.1%}, {r['wilson_hi']:.1%}]", axis=1)
disp["binom_p"] = disp["binom_p"].map(lambda x: f"{x:.3g}")
display(disp[["version", "judge", "pair", "fav", "wins_fav", "wins_other", "tie",
              "n_decided", "fav_share", "wilson", "binom_p", "sig_vs_coin"]]
        .sort_values(["version", "pair", "judge"]))

print("\nReading note:")
print("  • sig_vs_coin=True + fav=C-simple → independent judges significantly prefer baseline prose (R2/R4).")
print("  • sig_vs_coin=False on C-standard vs C-precedent → cannot reject coin-flip (supports R3 null).")
print("  • Self-judge often has huge tie counts → few decided votes → weak power (that's the R2 point).")

Pairwise decided-vote tests — H₀: P(fav)=½  |  Wilson 95% CI on fav share


,version,judge,pair,fav,wins_fav,wins_other,tie,n_decided,fav_share,wilson,binom_p,sig_vs_coin
10,v3,gpt-5.2 (self),C-simple_vs_C-precedent,C-simple,77,3,123,80,96.2%,"[89.5%, 98.7%]",1.41e-19,True
9,v3,gpt-5.2 (self),C-simple_vs_C-standard,C-simple,75,5,123,80,93.8%,"[86.2%, 97.3%]",4.25e-17,True
11,v3,GPT-5.5,C-standard_vs_C-precedent,C-standard,76,54,73,130,58.5%,"[49.9%, 66.6%]",0.0651,False
12,v3,Opus 4.8,C-standard_vs_C-precedent,C-standard,61,49,93,110,55.5%,"[46.1%, 64.4%]",0.294,False
13,v3,gpt-5.2 (self),C-standard_vs_C-precedent,C-standard,20,7,176,27,74.1%,"[55.3%, 86.8%]",0.0192,True
3,v4,GPT-5.5,C-simple_vs_C-precedent,C-simple,141,34,28,175,80.6%,"[74.1%, 85.8%]",1.1e-16,True
4,v4,Opus 4.8,C-simple_vs_C-precedent,C-simple,136,10,57,146,93.2%,"[87.9%, 96.2%]",2.14e-29,True
5,v4,gpt-5.2 (self),C-simple_vs_C-precedent,C-simple,29,16,158,45,64.4%,"[49.8%, 76.8%]",0.0725,False
0,v4,GPT-5.5,C-simple_vs_C-standard,C-simple,136,38,29,174,78.2%,"[71.5%, 83.7%]",3.86e-14,True
1,v4,Opus 4.8,C-simple_vs_C-standard,C-simple,119,15,69,134,88.8%,"[82.4%, 93.1%]",2.87e-21,True



Reading not<path>  • sig_vs_coin=True + fav=C-simple → independent judges significantly prefer baseline prose (R2/R4).
  • sig_vs_coin=False on C-standard vs C-precedent → cannot reject coin-flip (supports R3 null).
  • Self-judge often has huge tie counts → few decided votes → weak power (that's the R2 point).


In [ ]:
def _cohen_kappa(y1, y2, labels=None):
    """Cohen's κ for two raters, categorical labels."""
    y1 = np.asarray(y1)
    y2 = np.asarray(y2)
    if labels is Non<path>        labels = sorted(set(y1) | set(y2))
    lab_i = {l: i for i, l in enumerate(labels)}
    k = len(labels)
    n = len(y1)
    if n == 0 or k == 0:
        return np.nan
    mat = np.zeros((k, k), dtype=float)
    for a, b in zip(y1, y2):
        mat[lab_i[a], lab_i[b]] += 1
    po = np.trace(mat) / n
    pe = np.sum(mat.sum(0) * mat.sum(1)) / (n * n)
    if pe == 1:
        return 1.0 if po == 1 else np.nan
    return float((po - pe) / (1 - pe))

def kappa_bootstrap(version: str, n_boot: int = N_BOOT, alpha: float = 0.05) -> pd.DataFram<path>    raw = load_pairwise_results(version)
    # align on (doc_id, pair) across judges
    piv = raw.pivot_table(index=["doc_id", "pair"], columns="judge",
                          values="winner", aggfunc="first")
    rows = []
    judge_pairs = [("self", "opus"), ("self", "gpt55"), ("opus", "gpt55")]
    for pname, block in piv.groupby(level="pair"):
        for j1, j2 in judge_pair<path>            if j1 not in block.columns or j2 not in block.column<path>                continue
            sub = block[[j1, j2]].dropna()
            if len(sub) < 10:
                continue
            y1 = sub[j1].to_numpy()
            y2 = sub[j2].to_numpy()
            labels = sorted(set(y1) | set(y2))
            point = _cohen_kappa(y1, y2, labels)
            # bootstrap over items
            n = len(sub)
            boots = []
            for _ in range(n_boot):
                idx = RNG.integers(0, n, n)
                boots.append(_cohen_kappa(y1[idx], y2[idx], labels))
            lo, hi = np.nanpercentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
            rows.append({
                "version": version,
                "pair": pname,
                "judges": f"{j1}_vs_{j2}",
                "n": n,
                "kappa": point,
                "ci95_lo": float(lo),
                "ci95_hi": float(hi),
            })
    return pd.DataFrame(rows)

kappa_ci = pd.concat([kappa_bootstrap("v4"), kappa_bootstrap("v3")], ignore_index=True)
print("Pairwise Cohen's κ — bootstrap 95% CI")
disp = kappa_ci.copy()
disp["kappa"] = disp["kappa"].map(lambda x: f"{x:.3f}")
disp["ci95"] = disp.apply(lambda r: f"[{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]", axis=1)
display(disp[["version", "pair", "judges", "n", "kappa", "ci95"]]
        .sort_values(["version", "pair", "judges"]))

# R2 headline contrast: mean κ self↔opus vs opus↔gpt55 on v4
v4k = kappa_ci[kappa_ci["version"] == "v4"]
def _mean_k(key):
    s = v4k[v4k["judges"] == key]["kappa"]
    return float(s.mean()) if len(s) else np.nan
print(f"\nv4 mean κ across pairs: self↔opus={_mean_k('self_vs_opus'):.3f}, "
      f"self↔gpt55={_mean_k('self_vs_gpt55'):.3f}, "
      f"opus↔gpt55={_mean_k('opus_vs_gpt55'):.3f}")
print("R2 check: if self↔independent CIs sit below opus↔gpt55, the self-judge is the odd one out.")

Pairwise Cohen's κ — bootstrap 95% CI


,version,pair,judges,n,kappa,ci95
11,v3,C-standard_vs_C-precedent,opus_vs_gpt55,203,0.274,"[0.164, 0.378]"
10,v3,C-standard_vs_C-precedent,self_vs_gpt55,203,0.110,"[0.039, 0.183]"
9,v3,C-standard_vs_C-precedent,self_vs_opus,203,-0.003,"[-0.085, 0.074]"
2,v4,C-simple_vs_C-precedent,opus_vs_gpt55,203,0.222,"[0.105, 0.334]"
1,v4,C-simple_vs_C-precedent,self_vs_gpt55,203,0.078,"[0.024, 0.135]"
0,v4,C-simple_vs_C-precedent,self_vs_opus,203,0.068,"[0.004, 0.133]"
5,v4,C-simple_vs_C-standard,opus_vs_gpt55,203,0.186,"[0.077, 0.289]"
4,v4,C-simple_vs_C-standard,self_vs_gpt55,203,0.094,"[0.037, 0.152]"
3,v4,C-simple_vs_C-standard,self_vs_opus,203,0.118,"[0.049, 0.196]"
8,v4,C-standard_vs_C-precedent,opus_vs_gpt55,203,0.203,"[0.098, 0.306]"



v4 mean κ across pairs: self↔opus=0.113, self↔gpt55=0.098, opus↔gpt55=0.204
R2 check: if self↔independent CIs sit below opus↔gpt55, the self-judge is the odd one out.


In [ ]:
# Compact paper-ready figure: claim-precision deltas with 95% CI (v4)
v4ct = claim_tests[claim_tests["version"] == "v4"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, contrast, title in zip(
    axes,
    ["G0→G1 (std−simple)", "G1→G2 (prec−std)"],
    ["G0→G1 claim-precision Δ", "G1→G2 claim-precision Δ"],
):
    block = v4ct[v4ct["contrast"] == contrast].set_index("judge").reindex(JUDGES)
    x = np.arange(len(JUDGES))
    means = block["mean_delta"].to_numpy(dtype=float)
    yerr = np.vstack([means - block["ci95_lo"], block["ci95_hi"] - means])
    colors = [COLORS[j] for j in JUDGES]
    ax.bar(x, means, 0.55, color=colors, yerr=yerr, capsize=4,
           error_kw={"elinewidth": 1.2})
    ax.axhline(0, color="#333", lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([JUDGE_LABEL[j].replace(" ", "\n") for j in JUDGES], fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("Δ precision (paired)" if ax is axes[0] else "")
    # annotate CI-excludes-0
    for xi, (_, r) in enumerate(block.iterrows()):
        mark = "★" if r["ci_excludes_0"] else "ns"
        ax.text(xi, (r["ci95_hi"] if r["ci95_hi"] > 0 else r["ci95_lo"]) + 0.0015,
                mark, ha="center", fontsize=8, color="#BF616A" if r["ci_excludes_0"] else "#666")
fig.suptitle("v4: paired bootstrap 95% CI — ★ = CI excludes 0; ns = not significant", y=1.03)
fig.tight_layout()
plt.show()

# Pairwise fav-share with Wilson CI (v4, focused contrasts)
v4pw = focus_df[focus_df["version"] == "v4"]
pairs_show = list(dict.fromkeys(v4pw["pair"]))
fig, axes = plt.subplots(1, len(pairs_show), figsize=(4.0 * len(pairs_show), 4.2), sharey=True)
if len(pairs_show) == 1:
    axes = [axes]
for ax, pname in zip(axes, pairs_show):
    block = v4pw[v4pw["pair"] == pname].set_index("judge").reindex(
        [j for j in JUDGES if j in set(v4pw[v4pw["pair"] == pname]["judge"])]
    )
    fav = block["fav"].iloc[0]
    x = np.arange(len(block))
    means = block["fav_share"].to_numpy(dtype=float)
    yerr = np.vstack([means - block["wilson_lo"], block["wilson_hi"] - means])
    ax.bar(x, means, 0.55, color=[COLORS[j] for j in block.index],
           yerr=yerr, capsize=4, error_kw={"elinewidth": 1.2})
    ax.axhline(0.5, color="#333", lw=0.9, ls="--")
    ax.set_xticks(x)
    ax.set_xticklabels([JUDGE_LABEL[j].replace(" ", "\n") for j in block.index], fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{pname}\nfav={fav}")
    if ax is axes[0]:
        ax.set_ylabel("P(fav | decided)")
    for xi, (j, r) in enumerate(block.iterrows()):
        mark = "★" if r["sig_vs_coin"] else "ns"
        ax.text(xi, r["wilson_hi"] + 0.03, mark, ha="center", fontsize=8,
                color="#BF616A" if r["sig_vs_coin"] else "#666")
fig.suptitle("v4 pairwise: Wilson 95% CI on fav share among decided votes (★ p<0.05 vs coin-flip)", y=1.05)
fig.tight_layout()
plt.show()

## 6. What conclusions does this support?

Run the next cell to print a structured verdict table from the numbers above (not hand-waved).

In [ ]:
verdicts = []

def _ct(contrast, judge):
    r = claim_tests[(claim_tests["version"] == "v4") & (claim_tests["contrast"] == contrast)
                    & (claim_tests["judge"] == judge)]
    return r.iloc[0] if len(r) else None

def _pw(pair, fav, judge):
    r = focus_df[(focus_df["version"] == "v4") & (focus_df["pair"] == pair)
                 & (focus_df["fav"] == fav) & (focus_df["judge"] == judge)]
    return r.iloc[0] if len(r) else None

# --- R1: structure for free ---
_links_std = int(structure_links.loc[structure_links["tier"] == "C-standard", "authority_links_total"].iloc[0])
_lpc_std = float(structure_links.loc[structure_links["tier"] == "C-standard", "links_per_card"].iloc[0])
_prec_range = claim_summary[(claim_summary["version"] == "v4")]["claim_precision"]
_flat = claim_tests[claim_tests["version"] == "v4"]
# any *positive* lift (CI entirely above 0)?
_any_pos = bool(((_flat["ci95_lo"] > 0) & _flat["ci_excludes_0"]).any())
# any tiny negative (CI entirely below 0)?
_neg = _flat[(_flat["ci95_hi"] < 0) & _flat["ci_excludes_0"]]
_r1_note = (
    "no positive Δ CI; "
    + (f"{len(_neg)} tiny negative Δ CI(s) (see G0→G1 Opus)" if len(_neg) else "all CIs include 0")
)
verdicts.append({
    "claim": "R1: knowledge tiers add verifiable structure at no material faithfulness cost",
    "support": "SUPPORTED (deterministic + 3-judge + CI)",
    "evidence": (
        f"C-standard adds {_links_std} authority links ({_lpc_std}/card; baseline = 0), "
        f"identity-gated ({len(uniq_rej)} unique rejections incl. {reason_kind['lifespan conflict']} lifespan); "
        f"claim precision {_prec_range.min():.3f}–{_prec_range.max():.3f}; {_r1_note}"
    ),
})

# --- G1→G2 claim precision ---
bits = []
for j in JUDGE<path>    r = _ct("G1→G2 (prec−std)", j)
    bits.append(f"{j} {r['mean_delta']:+.4f} [{r['ci95_lo']:+.4f},{r['ci95_hi']:+.4f}]"
                f"{' ★' if r['ci_excludes_0'] else ' ns'}")
verdicts.append({
    "claim": "G1→G2 does not lift claim precision (v4)",
    "support": "SUPPORTED (all CIs include 0)" if not any(
        _ct("G1→G2 (prec−std)", j)["ci_excludes_0"] for j in JUDGES
    ) else "MIXED — see CIs",
    "evidence": "; ".join(bits),
})

# --- G0→G1 claim precision ---
bits = []
for j in JUDGE<path>    r = _ct("G0→G1 (std−simple)", j)
    wp = f" W-p={r['wilcoxon_p']:.3g}" if pd.notna(r["wilcoxon_p"]) else ""
    bits.append(f"{j} {r['mean_delta']:+.4f} [{r['ci95_lo']:+.4f},{r['ci95_hi']:+.4f}]"
                f"{' ★' if r['ci_excludes_0'] else ' ns'}{wp}")
_g01_pos = any(_ct("G0→G1 (std−simple)", j)["ci95_lo"] > 0 for j in JUDGES)
_g01_neg_star = any(
    _ct("G0→G1 (std−simple)", j)["ci_excludes_0"] and _ct("G0→G1 (std−simple)", j)["ci95_hi"] < 0
    for j in JUDGES
)
verdicts.append({
    "claim": "G0→G1 lifts claim precision (v4)",
    "support": (
        "NOT SUPPORTED — no positive lift; Opus finds tiny significant *decrease* (−0.4pp)"
        if _g01_neg_star and not _g01_pos
        else "NOT SUPPORTED (all CIs include 0)" if not _g01_pos
        else "PARTIAL — see CIs"
    ),
    "evidence": "; ".join(bits),
})

# --- Pairwise baseline preference ---
bits = []
for j in JUDGE<path>    r = _pw("C-simple_vs_C-standard", "C-simple", j)
    if r is Non<path>        continue
    bits.append(
        f"{j} {r['fav_share']:.0%} of decided "
        f"[{r['wilson_lo']:.0%},{r['wilson_hi']:.0%}] "
        f"p={r['binom_p']:.3g}{' ★' if r['sig_vs_coin'] else ' ns'} "
        f"(ties={r['tie']})"
    )
_ind_sig = all(
    _pw("C-simple_vs_C-standard", "C-simple", j) is not None
    and bool(_pw("C-simple_vs_C-standard", "C-simple", j)["sig_vs_coin"])
    for j in ("opus", "gpt55")
)
verdicts.append({
    "claim": "Independent judges prefer baseline over C-standard (pairwise, v4)",
    "support": "SUPPORTED (binomial ★ for opus+gpt55)" if _ind_sig else "MIXED",
    "evidence": "; ".join(bits),
})

# --- Pairwise G1 vs G2 ---
bits = []
_g12_any_sig = False
for j in JUDGE<path>    r = _pw("C-standard_vs_C-precedent", "C-standard", j)
    if r is Non<path>        continue
    if r["sig_vs_coin"]:
        _g12_any_sig = True
    bits.append(
        f"{j} {r['fav_share']:.0%} [{r['wilson_lo']:.0%},{r['wilson_hi']:.0%}] "
        f"p={r['binom_p']:.3g}{' ★' if r['sig_vs_coin'] else ' ns'}"
    )
verdicts.append({
    "claim": "G1→G2 changes holistic preference (v4)",
    "support": "NOT SUPPORTED (coin-flip; all ns)" if not _g12_any_sig else "MIXED — see binomial",
    "evidence": "; ".join(bits),
})

# --- Self-preference / judge validity ---
v4k = kappa_ci[kappa_ci["version"] == "v4"]
def _krow(key):
    s = v4k[v4k["judges"] == key]
    return float(s["kappa"].mean()), float(s["ci95_lo"].mean()), float(s["ci95_hi"].mean())
so_m, so_lo, so_hi = _krow("self_vs_opus")
og_m, og_lo, og_hi = _krow("opus_vs_gpt55")
sg_m, sg_lo, sg_hi = _krow("self_vs_gpt55")
verdicts.append({
    "claim": "Self-judge is interchangeable with cross-family judges on pairwise",
    "support": "REJECTED",
    "evidence": (
        f"mean κ self↔opus={so_m:.3f} [{so_lo:.3f},{so_hi:.3f}]; "
        f"self↔gpt55={sg_m:.3f} [{sg_lo:.3f},{sg_hi:.3f}]; "
        f"opus↔gpt55={og_m:.3f} [{og_lo:.3f},{og_hi:.3f}] — independents agree more with each other"
    ),
})

pdf, cdf, cl = agreement_tables("v4")
verdicts.append({
    "claim": "Claim-verify is stable across judge families (raw agreement)",
    "support": "SUPPORTED (with κ caveat)",
    "evidence": (
        f"self↔opus agree {cdf[cdf['judges']=='self_vs_opus']['agree_3way'].iloc[0]}% "
        f"on {cl['n_shared_claims']} claims; κ stays low because almost all claims are supported."
    ),
})

verdicts.append({
    "claim": "Bibliographic/completeness already saturate — need descriptive instruments",
    "support": "SUPPORTED",
    "evidence": (
        f"completeness ~{struct['completeness'].min():.2f}–{struct['completeness'].max():.2f}; "
        f"date exact ≥{struct['date_exact_rate'].min():.2f}; title overlap = 1.0 for all tiers."
    ),
})

verdict_df = pd.DataFrame(verdicts)
pd.set_option("display.max_colwidth", 220)
display(verdict_df)


,claim,support,evidence
0,R1: knowledge tiers add verifiable structure at no material faithfulness cost,SUPPORTED (deterministic + 3-judge + CI),"C-standard adds 2837 authority links (14.0/card; baseline = 0), identity-gated (361 unique rejections incl. 37 lifespan); claim precision 0.948–0.991; no positive Δ CI; 1 tiny negative Δ CI(s) (see G0→G1 Opus)"
1,G1→G2 does not lift claim precision (v4),SUPPORTED (all CIs include 0),"self -0.0043 [-0.0101,+0.0014] ns; opus +0.0022 [-0.0010,+0.0055] ns; gpt55 -0.0002 [-0.0126,+0.0114] ns"
2,G0→G1 lifts claim precision (v4),NOT SUPPORTED — no positive lift; Opus finds tiny significant *decrease* (−0.4pp),"self -0.0038 [-0.0094,+0.0017] ns W-p=0.142; opus -0.0041 [-0.0073,-0.0009] ★ W-p=0.00759; gpt55 -0.0104 [-0.0254,+0.0050] ns W-p=0.00986"
3,"Independent judges prefer baseline over C-standard (pairwise, v4)",SUPPORTED (binomial ★ for opus+gpt55),"self 58% of decided [43%,72%] p=0.36 ns (ties=160); opus 89% of decided [82%,93%] p=2.87e-21 ★ (ties=69); gpt55 78% of decided [71%,84%] p=3.86e-14 ★ (ties=29)"
4,G1→G2 changes holistic preference (v4),NOT SUPPORTED (coin-flip; all ns),"self 55% [35%,73%] p=0.832 ns; opus 51% [42%,61%] p=0.848 ns; gpt55 50% [42%,58%] p=1 ns"
5,Self-judge is interchangeable with cross-family judges on pairwise,REJECTED,"mean κ self↔opus=0.113 [0.043,0.187]; self↔gpt55=0.098 [0.039,0.159]; opus↔gpt55=0.204 [0.093,0.310] — independents agree more with each other"
6,Claim-verify is stable across judge families (raw agreement),SUPPORTED (with κ caveat),self↔opus agree 96.84% on 33450 claims; κ stays low because almost all claims are supported.
7,Bibliographic/completeness already saturate — need descriptive instruments,SUPPORTED,completeness ~0.92–0.94; date exact ≥0.99; title overlap = 1.0 for all tiers.


### Plain-language takeaway (reframed claims — see `docs/rag-design/09-claim-reframe-analysis.md`)

**With statistical tests (bootstrap 95% CI + binomial + Wilcoxon):**

1. **R1 — structure for free (headline, now CI-backed).** Knowledge grounding adds 2,837 identity-gated authority links (~14/card vs 0) while claim precision stays in 0.95–0.99. **No judge shows a positive precision lift** (all positive-side CIs fail). Opus alone finds a tiny significant *decrease* (−0.4pp, CI [−0.73, −0.09]pp, Wilcoxon p=0.008) — practically negligible on a saturated baseline, and the opposite of the old "quality ladder" claim.
2. **R2 — self-judging misleads (now binomial ★).** Among decided pairwise votes, Opus prefers baseline 89% (Wilson [82%, 93%], p≈10⁻²¹) and GPT-5.5 78% ([71%, 84%], p≈10⁻¹⁴); self is tie-heavy (160/203 ties) and **not** significant vs coin-flip (58%, p=0.36). Mean κ: self↔opus 0.11 vs opus↔gpt55 0.20.
3. **R3 — honest null on peer experience (CI + binomial).** G1→G2 claim-precision CIs all include 0; pairwise C-standard vs C-precedent is a coin-flip under every judge (50–55%, all ns).
4. **R4 — form vs richness.** Independent judges' significant baseline preference + thinner claim volume under LCSH (53 vs 59/card) = a real trade-off for the 30-issue human round to adjudicate.
5. **Dropped:** "G0→G1 lifts descriptive text quality" — not only unsupported; under Opus the point estimate goes the other way.

Paper tables should report the CIs / p-values above, not just means.
